# PixelPlayer (Sound of Pixels) AVSpeech Fine-Tuning

This notebook fine-tunes the pretrained Sound-of-Pixels model (originally trained on the MUSIC dataset) on the AVSpeech dataset to separate human speech.

**Hardware Target**: Google Colab (1x GPU, ~70GB Disk Limit)
**Strategy**: We use HuggingFace `datasets` streaming to download a manageable subset of AVSpeech (`MAX_SAMPLES=5000`) and format it exactly how the original `MUSICMixDataset` expects. This keeps our disk footprint well under 10GB while allowing the code to function unmodified.

## 1. Environment Setup

In [ ]:
!pip install torch torchvision torchaudio
!pip install librosa imageio soundfile mir_eval pyyaml 
!pip install datasets huggingface_hub
!apt-get install -y ffmpeg

## 2. Mount / Clone the Repository
Assuming you have uploaded the updated `Sound-of-Pixels` repository to your Colab environment or Google Drive.

In [ ]:
import os
# Change this to the path where your Sound-of-Pixels repo is uploaded
REPO_DIR = '/content/Sound-of-Pixels'

if not os.path.exists(REPO_DIR):
    print("Please ensure the Sound-of-Pixels repository is uploaded to Colab or clone it.")
    # !git clone https://github.com/hangzhaomit/Sound-of-Pixels {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Current working directory: {os.getcwd()}")

## 3. Download Pretrained MUSIC Weights
We initialize our networks with weights trained on the MUSIC dataset.

In [ ]:
!chmod +x ./scripts/download_trained_model.sh
!./scripts/download_trained_model.sh

## 4. Prepare AVSpeech Data (Streaming Subset to Disk)
We stream a subset of AVSpeech and save the video frames and audio to mimic the folder structure used by PixelPlayer.

In [ ]:
import os
import csv
import cv2
import numpy as np
import soundfile as sf
from datasets import load_dataset

DATA_DIR = './data/avspeech'
AUDIO_DIR = os.path.join(DATA_DIR, 'audio/speaker')
FRAMES_DIR = os.path.join(DATA_DIR, 'frames/speaker')

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)

MAX_SAMPLES = 5000  # Adjust based on time/disk availability
FPS = 8
SR_TARGET = 11025

print("Streaming AVSpeech dataset via HuggingFace...")
dataset = load_dataset("ProgramComputer/avspeech-visual-audio", split="train", streaming=True)

csv_rows = []
count = 0

def extract_frames(video_bytes, output_dir):
    # Write temp file for OpenCV to read
    temp_mp4 = "/tmp/temp_video.mp4"
    with open(temp_mp4, "wb") as f:
        f.write(video_bytes)
        
    cap = cv2.VideoCapture(temp_mp4)
    vid_fps = cap.get(cv2.CAP_PROP_FPS)
    if vid_fps <= 0: 
        vid_fps = 30.0 # fallback
        
    frame_interval = int(round(vid_fps / FPS))
    if frame_interval < 1: frame_interval = 1
    
    frame_idx = 0
    saved_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            saved_count += 1
            # PixelPlayer expects 1-indexed 6-digit padded names: 000001.jpg
            frame_name = os.path.join(output_dir, f"{saved_count:06d}.jpg")
            cv2.imwrite(frame_name, frame)
        frame_idx += 1
        
    cap.release()
    os.remove(temp_mp4)
    return saved_count

for item in dataset:
    if count >= MAX_SAMPLES:
        break
        
    clip_id = f"clip_{count:05d}"
    
    # 1. Save Audio
    # Note: HuggingFace uses different keys depending on the dataset structure.
    # The ProgramComputer/avspeech-visual-audio dataset often stores raw bytes or audio arrays.
    # Assuming 'audio' contains a numpy array and 'sampling_rate'
    try:
        audio_data = item['audio']['array']
        sr_original = item['audio']['sampling_rate']
        
        # It is handled via resampy or librosa normally, but we assume sf.write is OK
        # If the rate isn't 11025, base.py dataset loader will resample it on the fly anyway,
        # so we can just save it as is.
        audio_path = os.path.join(AUDIO_DIR, f"{clip_id}.wav")
        sf.write(audio_path, audio_data, sr_original)
        
    except Exception as e:
        print(f"Skipping audio for {clip_id}: {e}")
        continue
        
    # 2. Save Frames
    try:
        # Assuming 'video' contains raw raw mp4 bytes in HF dataset
        video_bytes = item['video']
        if isinstance(video_bytes, dict): 
            video_bytes = video_bytes.get('bytes', b'')
            
        frames_clip_dir = os.path.join(FRAMES_DIR, clip_id)
        os.makedirs(frames_clip_dir, exist_ok=True)
        
        num_frames = extract_frames(video_bytes, frames_clip_dir)
        
        if num_frames > 0:
            # 3. Append to CSV rows
            csv_rows.append([audio_path, frames_clip_dir, num_frames])
            count += 1
            
            if count % 100 == 0:
                print(f"Processed {count} / {MAX_SAMPLES} clips...")
                
    except Exception as e:
        print(f"Skipping video for {clip_id}: {e}")
        continue

print(f"Finished downloading and processing {count} clips.")

# Create Train / Val splits
import random
random.shuffle(csv_rows)
split_idx = int(0.9 * len(csv_rows))
train_rows = csv_rows[:split_idx]
val_rows = csv_rows[split_idx:]

with open(os.path.join(DATA_DIR, "train.csv"), "w", newline="") as f:
    csv.writer(f).writerows(train_rows)
    
with open(os.path.join(DATA_DIR, "val.csv"), "w", newline="") as f:
    csv.writer(f).writerows(val_rows)
    
print(f"Saved {len(train_rows)} train and {len(val_rows)} val index entries.")

## 5. Phase 1 Training: Finetune Visual Network Only
We adapt the visual features to recognize faces/mouths instead of instruments by freezing the audio network.

In [ ]:
!python -u main.py \
    --id AVSpeech-phase1 \
    --list_train ./data/avspeech/train.csv \
    --list_val ./data/avspeech/val.csv \
    --weights_sound ./ckpt/MUSIC_pretrained/net_sound_best.pth \
    --weights_frame ./ckpt/MUSIC_pretrained/net_frame_best.pth \
    --weights_synthesizer ./ckpt/MUSIC_pretrained/net_synthesizer_best.pth \
    --arch_frame resnet18dilated \
    --img_pool maxpool \
    --num_channels 32 \
    --binary_mask 1 \
    --loss bce \
    --weighted_loss 1 \
    --num_mix 2 \
    --log_freq 1 \
    --num_frames 3 \
    --stride_frames 24 \
    --frameRate 8 \
    --audLen 65535 \
    --audRate 11025 \
    --lr_frame 5e-5 \
    --lr_sound 0.0 \
    --lr_synthesizer 5e-4 \
    --num_epoch 15 \
    --lr_steps 10 \
    --num_gpus 1 \
    --workers 4 \
    --batch_size_per_gpu 16 \
    --disp_iter 20 \
    --num_vis 20 \
    --num_val 256

## 6. Phase 2 Training: Joint Finetuning
Now we unfreeze the audio network and adapt everything jointly for speech segregation.

In [ ]:
!python -u main.py \
    --id AVSpeech-phase2 \
    --list_train ./data/avspeech/train.csv \
    --list_val ./data/avspeech/val.csv \
    --weights_sound ./ckpt/AVSpeech-phase1/sound_best.pth \
    --weights_frame ./ckpt/AVSpeech-phase1/frame_best.pth \
    --weights_synthesizer ./ckpt/AVSpeech-phase1/synthesizer_best.pth \
    --arch_frame resnet18dilated \
    --img_pool maxpool \
    --num_channels 32 \
    --binary_mask 1 \
    --loss bce \
    --weighted_loss 1 \
    --num_mix 2 \
    --log_freq 1 \
    --num_frames 3 \
    --stride_frames 24 \
    --frameRate 8 \
    --audLen 65535 \
    --audRate 11025 \
    --lr_frame 1e-5 \
    --lr_sound 1e-4 \
    --lr_synthesizer 1e-4 \
    --num_epoch 35 \
    --lr_steps 15 25 \
    --num_gpus 1 \
    --workers 4 \
    --batch_size_per_gpu 16 \
    --disp_iter 20 \
    --num_vis 20 \
    --num_val 256

## 7. Evaluation
Run inference over the validation set and generate HTML visualizations.

In [ ]:
!python -u main.py \
    --mode eval \
    --id AVSpeech-eval \
    --weights_sound ./ckpt/AVSpeech-phase2/sound_best.pth \
    --weights_frame ./ckpt/AVSpeech-phase2/frame_best.pth \
    --weights_synthesizer ./ckpt/AVSpeech-phase2/synthesizer_best.pth \
    --list_val ./data/avspeech/val.csv \
    --num_mix 2 \
    --audRate 11025 \
    --audLen 65535

import IPython
IPython.display.HTML(filename='./ckpt/AVSpeech-eval/visualization/index.html')